qsub run_exp5_variable_baseline_physical.pbs

# CMIP variable prediction — Physical Baseline

### Data preprocessing

**Library import**

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
import torch
import torch.nn as nn
import json
from pathlib import Path

EXPERIENCE main HYPERPARAMETERS     :

In [ ]:
setup_name = "baseline_physical"

In [ ]:
num_sample = 2500000
variable = "pr"
val_fraction = 0.05
test_fraction = 0.15

In [ ]:
# ========================================
# Baseline predictor on raw features
# ========================================

baseline_n_epochs = 100

baseline_batch_size = 256
baseline_patience = 10
baseline_learning_rate = 1e-3
baseline_weight_decay = 1e-3
baseline_hidden_dim = 128
baseline_n_hidden_layers = 5

**Data Loading**

You have to run a PBS job to create the data loaded in the next cell. In the pbs file you can choose the following parameters :

- --max-abs-lat value \ (recommanded : 30)
- --patch-size-km value \ (recommanded : 1000)
- --time-stride value \ (recommanded : 24)
- --max-samples-per-climate value \ (recommanded : 1000)
- --random-seed value \ (recommanded : 42)

to run the PBS job use the following command in /glade/u/home/tsalin/CMIP: 

qsub run_build_multivariate_samples.pbs

to follow what's going on : 

qstat -u tsalin

In [ ]:
precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NGS_v{num_sample}")

if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])
max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])

features_by_climate_full = {
    c: np.load(precomputed_dir / f"features_{c}.npy")
    for c in climate_order
}
metadata_by_climate = {
    c: pd.read_csv(precomputed_dir / f"metadata_{c}.csv")
    for c in climate_order
}

# Split off the specified variable as a dedicated label while keeping sample/grid-point alignment.
if variable not in selected_variables_full:
    raise ValueError(f"Variable '{variable}' was not found in run_config selected_variables.")

n_variables_full = len(selected_variables_full)
expected_dim_full = n_variables_full * grid_points_per_patch
variable_var_index = selected_variables_full.index(variable)
variable_col_start = variable_var_index * grid_points_per_patch
variable_col_end = variable_col_start + grid_points_per_patch

feature_mask_without_variable = np.ones(expected_dim_full, dtype=bool)
feature_mask_without_variable[variable_col_start:variable_col_end] = False

label_variable_by_climate = {}
features_by_climate = {}
for c in climate_order:
    X_full = features_by_climate_full[c]
    if X_full.shape[1] != expected_dim_full:
        raise ValueError(
            f"Unexpected feature dimension for {c}: got {X_full.shape[1]}, expected {expected_dim_full}."
        )

    # Keep the specified variable values (one value per patch grid point) as label for each sample.
    label_variable_by_climate[c] = X_full[:, variable_col_start:variable_col_end].copy()

    # Keep all non-variable variables as model features used in the rest of the notebook.
    features_by_climate[c] = X_full[:, feature_mask_without_variable]

selected_variables = [v for v in selected_variables_full if v != variable]

# Convenience aggregate preserving same sample order as stacked climate features.
label_variable = np.vstack([label_variable_by_climate[c] for c in climate_order])

sample_count_df = pd.read_csv(precomputed_dir / "sample_count.csv").set_index("scenario")
pre_sampling_df = pd.read_csv(precomputed_dir / "pre_sampling_df.csv").set_index("scenario")
patch_catalog = pd.read_csv(precomputed_dir / "patch_catalog.csv")
sampling_diagnostics_df = pd.read_csv(precomputed_dir / "sampling_diagnostics.csv").set_index("scenario")
nan_summary_by_variable_df = pd.read_csv(precomputed_dir / "nan_summary_by_variable.csv")
sample_pairs_by_climate = {
    c: pd.read_csv(precomputed_dir / f"sample_pairs_{c}.csv")
    for c in climate_order
}

display(sample_count_df)
display(pre_sampling_df)
display(sampling_diagnostics_df)
print(f"Feature variables used downstream (without {variable}): {selected_variables}")
print(f"label_{variable} shape (all samples x grid points): {label_variable.shape}")

In [ ]:
# Backward-compatible aliases used later in the notebook
climate_order = list(climate_order)
climate_colors = dict(climate_colors)
sample_count_df = sample_count_df.copy()
pre_sampling_df = pre_sampling_df.copy()
patch_catalog = patch_catalog.copy()
sampling_diagnostics_df = sampling_diagnostics_df.copy()
nan_summary_by_variable_df = nan_summary_by_variable_df.copy()
baseline_train_climates = ["historical"]

**Latitude Band, 1000-km Patches, and Multivariate Samples**

This step builds one unified multivariate dataset used by all downstream analyses.

Workflow:
- select data inside +/- max_abs_lat (default: 30 deg);
- split the region into non-overlapping ~1000 km x 1000 km patches;
- for each climate, build samples from all variables and all lat/lon points inside each patch at sampled times;
- aggregate all climates to fit one common scaler

Utilities :

In [ ]:
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Train/test split

In [ ]:
def build_split_indices(data_by_climate, val_fraction=0.10, test_fraction=0.20, seed=42):
    """Build train/val/test split indices for each climate before any standardization."""
    split_indices = {}
    rng = np.random.default_rng(seed)

    for climate, X in data_by_climate.items():
        n = X.shape[0]
        indices = np.arange(n)
        rng.shuffle(indices)

        n_test = max(1, int(round(test_fraction * n)))
        n_val = max(1, int(round(val_fraction * n)))
        n_train = max(1, n - n_val - n_test)

        train_idx = indices[:n_train]
        val_idx = indices[n_train:n_train + n_val]
        test_idx = indices[n_train + n_val:]

        split_indices[climate] = {
            "train": train_idx,
            "val": val_idx,
            "test": test_idx,
        }

    return split_indices


# Build splits on raw, unstandardized data first.
ae_split_indices = build_split_indices(
    features_by_climate,
    val_fraction=val_fraction,
    test_fraction=test_fraction,
    seed=random_seed,
)

# ── RAM optimization: truncate eval-only climates to their test split ───────
_eval_only_climates = [c for c in climate_order if c not in baseline_train_climates]
if _eval_only_climates:
    for c in _eval_only_climates:
        _idx = ae_split_indices[c]["test"]
        _X_sub = features_by_climate[c][_idx]
        features_by_climate[c] = _X_sub
        _y_sub = label_variable_by_climate[c][_idx]
        label_variable_by_climate[c] = _y_sub
        features_by_climate_full[c] = features_by_climate_full[c][_idx]
        metadata_by_climate[c] = metadata_by_climate[c].iloc[_idx].reset_index(drop=True)
        ae_split_indices[c] = {
            "train": np.array([], dtype=np.intp),
            "val":   np.array([], dtype=np.intp),
            "test":  np.arange(len(_idx), dtype=np.intp),
        }
    print(f"RAM opt: {_eval_only_climates} truncated to {len(_idx)} samples (test split).")
    del _idx, _X_sub, _y_sub
del _eval_only_climates
# ── End RAM optimization ──────────────────────────────────────────────────────


# ==============================================================================
# Physical and statistical feature computation (from raw, unscaled data)
# ==============================================================================

n_input_variables = len(selected_variables)


def compute_rh(hus, ta, p):
    eps = 0.622
    e  = (hus * p) / (eps + (1.0 - eps) * hus)
    es = 611.2 * np.exp(17.67 * (ta - 273.15) / (ta - 29.65))
    return e / es


def get_raw_var(X_raw, var_name):
    vi = selected_variables.index(var_name)
    s  = slice(vi * grid_points_per_patch, (vi + 1) * grid_points_per_patch)
    return np.asarray(X_raw[:, s], dtype=np.float64)


# Build monthly-patch climatology from historical train samples only
hist_train_idx  = ae_split_indices["historical"]["train"]
hist_meta_train = metadata_by_climate["historical"].iloc[hist_train_idx].reset_index(drop=True)
X_hist_for_clim = np.asarray(features_by_climate["historical"][hist_train_idx], dtype=np.float64)

from collections import defaultdict

_clim_sums    = defaultdict(lambda: np.zeros(n_input_variables * grid_points_per_patch, dtype=np.float64))
_clim_counts  = defaultdict(int)
_month_sums   = defaultdict(lambda: np.zeros(n_input_variables * grid_points_per_patch, dtype=np.float64))
_month_counts = defaultdict(int)

for _i in range(len(hist_train_idx)):
    _pid = int(hist_meta_train["patch_id"].iloc[_i])
    _m   = int(hist_meta_train["month"].iloc[_i])
    _clim_sums[(_pid, _m)]   += X_hist_for_clim[_i]
    _clim_counts[(_pid, _m)] += 1
    _month_sums[_m]           += X_hist_for_clim[_i]
    _month_counts[_m]         += 1

clim_mean_patch_month = {k: _clim_sums[k] / _clim_counts[k] for k in _clim_sums}
clim_mean_monthly     = {m: _month_sums[m] / _month_counts[m] for m in _month_sums}

del _clim_sums, _clim_counts, _month_sums, _month_counts, X_hist_for_clim
print(f"Climatology: {len(clim_mean_patch_month)} (patch, month) pairs from {len(hist_train_idx)} hist_train samples.")

physical_variable_names = [
    "RH_surface", "RH850", "RH500",
    "lapse_850_500", "surface_850_temp_diff",
    "shear_u", "shear_v", "shear_mag",
    "wind850", "wind500",
] + [f"{v}_anom" for v in selected_variables]
n_physical_variables = len(physical_variable_names)  # 25


def compute_physical_features(X_raw, metadata):
    """Compute physical/statistical features from raw (unscaled) inputs.
    Returns array of shape (n_samples, 25 * grid_points_per_patch).
    Order: RH_surface, RH850, RH500, lapse_850_500, surface_850_temp_diff,
           shear_u, shear_v, shear_mag, wind850, wind500, then 15 anomalies.
    """
    X_raw = np.asarray(X_raw, dtype=np.float64)
    n_s   = X_raw.shape[0]
    gpp   = grid_points_per_patch
    out   = np.empty((n_s, n_physical_variables * gpp), dtype=np.float64)

    tas    = get_raw_var(X_raw, "tas")
    ta850  = get_raw_var(X_raw, "ta850")
    ta500  = get_raw_var(X_raw, "ta500")
    huss   = get_raw_var(X_raw, "huss")
    hus850 = get_raw_var(X_raw, "hus850")
    hus500 = get_raw_var(X_raw, "hus500")
    ua850  = get_raw_var(X_raw, "ua850")
    va850  = get_raw_var(X_raw, "va850")
    ua500  = get_raw_var(X_raw, "ua500")
    va500  = get_raw_var(X_raw, "va500")
    psl    = get_raw_var(X_raw, "psl")

    out[:, 0*gpp:1*gpp]  = compute_rh(huss,   tas,   psl)
    out[:, 1*gpp:2*gpp]  = compute_rh(hus850, ta850, 85000.0)
    out[:, 2*gpp:3*gpp]  = compute_rh(hus500, ta500, 50000.0)
    out[:, 3*gpp:4*gpp]  = ta850 - ta500
    out[:, 4*gpp:5*gpp]  = tas   - ta850
    du = ua500 - ua850
    dv = va500 - va850
    out[:, 5*gpp:6*gpp]  = du
    out[:, 6*gpp:7*gpp]  = dv
    out[:, 7*gpp:8*gpp]  = np.sqrt(du**2 + dv**2)
    out[:, 8*gpp:9*gpp]  = np.sqrt(ua850**2 + va850**2)
    out[:, 9*gpp:10*gpp] = np.sqrt(ua500**2 + va500**2)

    patch_ids  = metadata["patch_id"].values
    months     = metadata["month"].values
    _zero_clim = np.zeros(n_input_variables * gpp, dtype=np.float64)
    for _i in range(n_s):
        _key  = (int(patch_ids[_i]), int(months[_i]))
        _clim = clim_mean_patch_month.get(_key, clim_mean_monthly.get(int(months[_i]), _zero_clim))
        out[_i, 10*gpp:] = X_raw[_i] - _clim

    return out.astype(np.float32)


raw_physical_features_by_climate = {
    c: compute_physical_features(features_by_climate[c], metadata_by_climate[c])
    for c in climate_order
}
print("Raw physical features computed:")
for c in climate_order:
    print(f"  {c}: {raw_physical_features_by_climate[c].shape}")


# ==============================================================================
# Normalization — fit on historical train, apply to all climates
# ==============================================================================

reference_climate = "historical"
if reference_climate not in features_by_climate:
    raise KeyError(f"Reference climate {reference_climate!r} not found in features_by_climate.")

X_hist_train_raw = np.asarray(features_by_climate[reference_climate][hist_train_idx], dtype=np.float32)
y_hist_train_raw = np.asarray(label_variable_by_climate[reference_climate][hist_train_idx], dtype=np.float32)
P_hist_train_raw = np.asarray(raw_physical_features_by_climate[reference_climate][hist_train_idx], dtype=np.float32)

expected_input_dim = n_input_variables * grid_points_per_patch
if X_hist_train_raw.shape[1] != expected_input_dim:
    raise ValueError(
        f"Unexpected input dimension: got {X_hist_train_raw.shape[1]}, "
        f"expected {expected_input_dim} = {n_input_variables} variables x {grid_points_per_patch} points."
    )
if y_hist_train_raw.shape[1] != grid_points_per_patch:
    raise ValueError(
        f"Unexpected label dimension: got {y_hist_train_raw.shape[1]}, expected {grid_points_per_patch}."
    )

# ── Raw input variable normalization ──────────────────────────────────────────
input_variable_means = np.zeros(n_input_variables, dtype=np.float64)
input_variable_stds  = np.ones(n_input_variables,  dtype=np.float64)

for var_idx, var_name in enumerate(selected_variables):
    cols_for_var = slice(
        var_idx * grid_points_per_patch,
        (var_idx + 1) * grid_points_per_patch,
    )
    values = X_hist_train_raw[:, cols_for_var].reshape(-1)
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError(f"No finite historical train values found for input variable {var_name!r}.")
    if var_name == "pr":
        values = np.log1p(values * 86400)
    mu    = float(np.mean(values))
    sigma = float(np.std(values))
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0
    input_variable_means[var_idx] = mu
    input_variable_stds[var_idx]  = sigma

# ── Label normalization ────────────────────────────────────────────────────────
label_values = y_hist_train_raw.reshape(-1)
label_values = label_values[np.isfinite(label_values)]
if label_values.size == 0:
    raise ValueError(f"No finite historical train values found for label variable {variable!r}.")
if variable == "pr":
    label_values = np.log1p(label_values * 86400)
label_mean = float(np.mean(label_values))
label_std  = float(np.std(label_values))
if not np.isfinite(label_std) or label_std <= 0:
    label_std = 1.0

# ── Physical feature normalization ────────────────────────────────────────────
physical_variable_means = np.zeros(n_physical_variables, dtype=np.float64)
physical_variable_stds  = np.ones(n_physical_variables,  dtype=np.float64)

for phys_idx in range(n_physical_variables):
    cols   = slice(phys_idx * grid_points_per_patch, (phys_idx + 1) * grid_points_per_patch)
    values = P_hist_train_raw[:, cols].reshape(-1)
    values = values[np.isfinite(values)]
    if values.size == 0:
        continue
    mu    = float(np.mean(values))
    sigma = float(np.std(values))
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0
    physical_variable_means[phys_idx] = mu
    physical_variable_stds[phys_idx]  = sigma

# ── Normalization functions ────────────────────────────────────────────────────
def standardize_input_variables(X_raw):
    X_raw    = np.asarray(X_raw, dtype=np.float64)
    X_scaled = np.empty_like(X_raw, dtype=np.float32)
    for var_idx, var_name in enumerate(selected_variables):
        cols_for_var = slice(
            var_idx * grid_points_per_patch,
            (var_idx + 1) * grid_points_per_patch,
        )
        values = X_raw[:, cols_for_var]
        if var_name == "pr":
            values = np.log1p(values * 86400)
        X_scaled[:, cols_for_var] = (
            (values - input_variable_means[var_idx]) / input_variable_stds[var_idx]
        ).astype(np.float32)
    return X_scaled


def standardize_physical_features(P_raw):
    P_raw    = np.asarray(P_raw, dtype=np.float64)
    P_scaled = np.empty_like(P_raw, dtype=np.float32)
    for phys_idx in range(n_physical_variables):
        cols = slice(phys_idx * grid_points_per_patch, (phys_idx + 1) * grid_points_per_patch)
        P_scaled[:, cols] = (
            (P_raw[:, cols] - physical_variable_means[phys_idx]) / physical_variable_stds[phys_idx]
        ).astype(np.float32)
    return P_scaled


def standardize_label_variable(y_raw):
    y_raw = np.asarray(y_raw, dtype=np.float64)
    if variable == "pr":
        y_raw = np.log1p(y_raw * 86400)
    return ((y_raw - label_mean) / label_std).astype(np.float32)


def denormalize_label_variable(y_scaled):
    y_scaled = np.asarray(y_scaled, dtype=np.float64)
    result   = y_scaled * label_std + label_mean
    if variable == "pr":
        result = np.expm1(result)
    return result.astype(np.float32)


# ── Apply normalization and concatenate ───────────────────────────────────────
scaled_features_by_climate          = {}
scaled_label_variable_by_climate    = {}
scaled_physical_features_by_climate = {}
augmented_features_by_climate       = {}

for climate in climate_order:
    X_raw = np.asarray(features_by_climate[climate],              dtype=np.float32)
    y_raw = np.asarray(label_variable_by_climate[climate],        dtype=np.float32)
    P_raw = np.asarray(raw_physical_features_by_climate[climate], dtype=np.float32)

    X_sc = standardize_input_variables(X_raw)
    P_sc = standardize_physical_features(P_raw)

    scaled_features_by_climate[climate]          = X_sc
    scaled_label_variable_by_climate[climate]    = standardize_label_variable(y_raw)
    scaled_physical_features_by_climate[climate] = P_sc
    augmented_features_by_climate[climate]       = np.concatenate([X_sc, P_sc], axis=1)

label_variable_by_climate = scaled_label_variable_by_climate
label_variable = np.vstack([label_variable_by_climate[c] for c in climate_order])

aug_dim = next(iter(augmented_features_by_climate.values())).shape[1]
print(f"\n✓ Augmented feature dimension: {aug_dim}  (1050 raw + {aug_dim - 1050} physical)")
print("Augmented shapes:", {c: augmented_features_by_climate[c].shape for c in climate_order})

In [ ]:
# RAM Reduction
del features_by_climate_full
del features_by_climate
del raw_physical_features_by_climate
del scaled_physical_features_by_climate

**Predictor architecture and data loader utility** (same MLP structure as CERA predictors, reused here for a fair comparison):

In [ ]:
# CERA-like predictive invariant AE: Classes and Helper Functions

class CERAPredictor(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int = 128, n_hidden_layers: int = 5):
        super().__init__()
        layers = []
        in_dim = input_dim
        for _ in range(n_hidden_layers):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.LeakyReLU(negative_slope=0.1))
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

def cera_make_labeled_loader(
    data_by_climate,
    labels_by_climate,
    split_indices,
    climate: str,
    split: str,
    batch_size: int,
    shuffle: bool,
    drop_last: bool,
 ):
    idx = split_indices[climate][split]
    X = np.asarray(data_by_climate[climate][idx], dtype=np.float32)
    y = np.asarray(labels_by_climate[climate][idx], dtype=np.float32)
    ds = torch.utils.data.TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)

### Training and evaluating a baseline

Baseline training :

In [ ]:
# Checkpoint: path to save/resume training state between PBS jobs
baseline_checkpoint_path = Path("/glade/u/home/tsalin/CMIP/model_evaluation/Baseline_physical/baseline_physical_checkpoint.pt")
baseline_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
baseline_checkpoint_freq = 1  # Save checkpoint every N epochs (1 = every epoch)


def train_baseline_physical_predictor(
    train_climates=None,
    n_epochs: int = 100,
    lr: float = 1e-3,
    batch_size: int = 256,
    weight_decay: float = 1e-5,
    hidden_dim: int = 128,
    n_hidden_layers: int = 5,
    patience: int = 10,
    checkpoint_path=None,
    checkpoint_freq=1,
):
    """
    Train a physical-feature MLP predictor from augmented scaled features -> label target.

    Architecture matches CERAPredictor in depth/width.
    Training uses only historical samples (train/val).
    Input: augmented_features_by_climate (raw scaled + physical scaled).
    """
    if train_climates is None:
        train_climates = ["historical"]
    if train_climates != ["historical"]:
        raise ValueError("This setup expects train_climates=['historical']")

    input_dim_local = next(iter(augmented_features_by_climate.values())).shape[1]
    output_dim_pred = int(grid_points_per_patch)

    baseline_model = CERAPredictor(
        input_dim=input_dim_local,
        output_dim=output_dim_pred,
        hidden_dim=hidden_dim,
        n_hidden_layers=n_hidden_layers,
    ).to(device)

    optimizer = torch.optim.AdamW(baseline_model.parameters(), lr=lr, weight_decay=weight_decay)
    mse_loss_fn = nn.MSELoss()

    train_loader_hist = cera_make_labeled_loader(
        augmented_features_by_climate,
        label_variable_by_climate,
        ae_split_indices,
        "historical",
        "train",
        batch_size,
        True,
        True,
    )
    val_loader_hist = cera_make_labeled_loader(
        augmented_features_by_climate,
        label_variable_by_climate,
        ae_split_indices,
        "historical",
        "val",
        batch_size,
        False,
        True,
    )

    best_val = np.inf
    best_epoch = -1
    patience_counter = 0
    best_model_state = None
    history = []

    start_epoch = 1
    if checkpoint_path is not None:
        _ckpt_path = Path(checkpoint_path)
        if _ckpt_path.exists():
            _ckpt = torch.load(_ckpt_path, map_location=device)
            baseline_model.load_state_dict(_ckpt["model_state"])
            optimizer.load_state_dict(_ckpt["optimizer_state"])
            start_epoch       = _ckpt["epoch"] + 1
            best_val          = _ckpt["best_val"]
            best_epoch        = _ckpt["best_epoch"]
            patience_counter  = _ckpt["patience_counter"]
            best_model_state  = _ckpt["best_model_state"]
            history           = _ckpt["history"]
            print(f"[CHECKPOINT] Resuming from epoch {start_epoch} "
                  f"(best: epoch {best_epoch}, val={best_val:.6f})")

    for epoch in range(start_epoch, n_epochs + 1):
        baseline_model.train()
        train_losses = []

        for xb_hist, yb_hist in train_loader_hist:
            xb = xb_hist.to(device)
            yb = yb_hist.to(device)

            optimizer.zero_grad()
            y_pred = baseline_model(xb)
            loss = mse_loss_fn(y_pred, yb)
            loss.backward()
            optimizer.step()

            train_losses.append(float(loss.detach().cpu().item()))

        baseline_model.eval()
        val_losses = []

        with torch.no_grad():
            for xb_hist, yb_hist in val_loader_hist:
                xb = xb_hist.to(device)
                yb = yb_hist.to(device)

                y_pred = baseline_model(xb)
                loss = mse_loss_fn(y_pred, yb)
                val_losses.append(float(loss.detach().cpu().item()))

        train_loss = float(np.mean(train_losses)) if train_losses else np.nan
        val_loss = float(np.mean(val_losses)) if val_losses else np.nan

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
            }
        )

        print(f"Epoch {epoch}/{n_epochs} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} "f"(best_val={best_val:.4f} @ {best_epoch})")

        if checkpoint_path is not None and epoch % checkpoint_freq == 0:
            torch.save({
                "epoch": epoch,
                "model_state": {k: v.cpu().clone() for k, v in baseline_model.state_dict().items()},
                "optimizer_state": optimizer.state_dict(),
                "best_val": best_val,
                "best_epoch": best_epoch,
                "patience_counter": patience_counter,
                "best_model_state": best_model_state,
                "history": history,
            }, checkpoint_path)

        if val_loss < best_val:
            best_val = val_loss
            best_epoch = epoch
            best_model_state = {k: v.detach().cpu().clone() for k, v in baseline_model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if checkpoint_path is not None:
                    torch.save({
                        "epoch": epoch,
                        "model_state": {k: v.cpu().clone() for k, v in baseline_model.state_dict().items()},
                        "optimizer_state": optimizer.state_dict(),
                        "best_val": best_val,
                        "best_epoch": best_epoch,
                        "patience_counter": patience_counter,
                        "best_model_state": best_model_state,
                        "history": history,
                    }, checkpoint_path)
                break

    if best_model_state is not None:
        baseline_model.load_state_dict(best_model_state)

    history_df = pd.DataFrame(history)
    return baseline_model, history_df, best_epoch


baseline_predictor, baseline_history_df, baseline_best_epoch = train_baseline_physical_predictor(
    train_climates=baseline_train_climates,
    n_epochs=baseline_n_epochs,
    lr=baseline_learning_rate,
    batch_size=baseline_batch_size,
    weight_decay=baseline_weight_decay,
    hidden_dim=baseline_hidden_dim,
    n_hidden_layers=baseline_n_hidden_layers,
    patience=baseline_patience,
    checkpoint_path=baseline_checkpoint_path,
    checkpoint_freq=baseline_checkpoint_freq,
)

print("Baseline training climates:", ", ".join(baseline_train_climates))
print("Baseline best epoch:", baseline_best_epoch)

display(baseline_history_df.tail())

# Backward-compatible explicit output name
baseline_physical_history_df = baseline_history_df

### Baseline evaluation - EXTRACTION

In [ ]:
component_order = {"prediction": 0}


def _build_meta_df(component, climate, metadata):
    """Build a lightweight metadata DataFrame without storing numerical arrays per row."""
    df = metadata.copy().reset_index(drop=True)
    df = df.drop(columns=["scenario"], errors="ignore")
    df.insert(0, "experiment",        "CMIP_variable_exp5_baseline_physical")
    df.insert(1, "component",         component)
    df.insert(2, "component_order",   component_order[component])
    df.insert(3, "scenario",          climate)
    df.insert(4, "scenario_order",    int(climate_order.index(climate)))
    df.insert(5, "sample_idx",        range(len(df)))
    return df


prediction_value_names = [
    f"{variable}@point_{point_idx}"
    for point_idx in range(grid_points_per_patch)
]

meta_dfs_pred  = []
truth_arrays_pred = []
pred_arrays_pred  = []

baseline_predictor.eval()
for climate in climate_order:
    idx = ae_split_indices[climate]["test"]
    X = np.asarray(augmented_features_by_climate[climate][idx], dtype=np.float32)
    y_true_scaled = np.asarray(label_variable_by_climate[climate][idx], dtype=np.float32)
    metadata = metadata_by_climate[climate].iloc[idx].reset_index(drop=True)

    with torch.inference_mode():
        xb = torch.from_numpy(X).to(device)
        y_pred_scaled = baseline_predictor(xb)
        y_hat_scaled = y_pred_scaled.detach().cpu().numpy()

    # Store final evaluation values in physical units.
    y_true_physical = denormalize_label_variable(y_true_scaled)
    y_hat_physical = denormalize_label_variable(y_hat_scaled)

    meta_dfs_pred.append(_build_meta_df("prediction", climate, metadata))
    truth_arrays_pred.append(y_true_physical.astype(np.float32))
    pred_arrays_pred.append(y_hat_physical.astype(np.float32))

baseline_physical_quality_payload = {
    "meta_prediction":        pd.concat(meta_dfs_pred,  ignore_index=True),
    "truth_prediction":       np.concatenate(truth_arrays_pred, axis=0),
    "pred_prediction":        np.concatenate(pred_arrays_pred,  axis=0),
    "prediction_value_names": prediction_value_names,
}

print("Baseline physical quality payload — shapes:")
print(f"  meta_prediction  : {baseline_physical_quality_payload['meta_prediction'].shape}")
print(f"  truth_prediction : {baseline_physical_quality_payload['truth_prediction'].shape}")
print(f"  pred_prediction  : {baseline_physical_quality_payload['pred_prediction'].shape}")
print()
print(baseline_physical_quality_payload["meta_prediction"].groupby("scenario").size().rename("n_samples_prediction"))

In [ ]:
# Delete the checkpoint once the whole notebook has run successfully
if baseline_checkpoint_path.exists():
    baseline_checkpoint_path.unlink()
    print(f"[CHECKPOINT] Checkpoint deleted: {baseline_checkpoint_path}")
else:
    print("[CHECKPOINT] No checkpoint to delete.")